In [0]:
from pyspark.sql import functions as F


# ============================================================
# CONFIGURATION
# ============================================================

SILVER = "ecommerce_returns.silver"
GOLD = "ecommerce_returns.gold"


# ============================================================
# 1. DIM_CUSTOMER
# ============================================================

customers = spark.table(f"{SILVER}.customers")

dim_customer = (
    customers
    .select(
        "customer_id",
        "customer_name",
        "city",
        "customer_segment"
    )
    .dropDuplicates(["customer_id"])
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.dim_customer")
)

print("Created gold.dim_customer")


# ============================================================
# 2. DIM_PRODUCT
# ============================================================

products = spark.table(f"{SILVER}.products")

dim_product = (
    products
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "seller_id",
        "cost_price",
        "selling_price",
        "weight_grams"
    )
    .dropDuplicates(["product_id"])
)

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.dim_product")
)

print("Created gold.dim_product")


# ============================================================
# 3. DIM_SELLER
# ============================================================

sellers = spark.table(f"{SILVER}.sellers")

dim_seller = (
    sellers
    .select(
        "seller_id",
        "seller_name",
        "seller_region",
        "seller_rating",
        "seller_join_date"
    )
    .dropDuplicates(["seller_id"])
)

(
    dim_seller.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.dim_seller")
)

print("Created gold.dim_seller")


# ============================================================
# 4. DIM_DATE
# ============================================================

orders = spark.table(f"{SILVER}.orders")

date_range = orders.select(
    F.min("order_date").alias("min_date"),
    F.max("order_date").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS date
        """
    )
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week", F.weekofyear("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.dim_date")
)

print("Created gold.dim_date")


# ============================================================
# 5. FACT_ORDERS
# ============================================================

fact_orders = (
    orders
    .join(
        dim_product.select(
            "product_id",
            "category",
            "subcategory",
            "brand"
        ),
        "product_id",
        "left"
    )
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "seller_id",
        "order_date",
        "quantity",
        "unit_price",
        "discount",
        "payment_method",
        "order_status",
        "category",
        "subcategory",
        "brand"
    )
    .withColumn(
        "gross_amount",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
    .withColumn(
        "discount_amount",
        F.round(
            F.col("quantity") *
            F.col("unit_price") *
            (F.col("discount") / 100),
            2
        )
    )
    .withColumn(
        "net_order_amount",
        F.round(
            F.col("gross_amount") -
            F.col("discount_amount"),
            2
        )
    )
)

(
    fact_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.fact_orders")
)

print("Created gold.fact_orders")


# ============================================================
# 6. FACT_RETURNS
# ============================================================

returns = spark.table(f"{SILVER}.returns")

fact_returns = (
    returns
    .join(
        fact_orders.select(
            "order_id",
            "order_date",
            "unit_price"
        ),
        "order_id",
        "left"
    )
    .withColumn(
        "return_amount",
        F.round(
            F.col("return_quantity") *
            F.col("unit_price"),
            2
        )
    )
    .withColumn(
        "return_days_after_order",
        F.datediff(
            F.col("return_date"),
            F.col("order_date")
        )
    )
    .select(
        "return_id",
        "order_id",
        "product_id",
        "customer_id",
        "return_date",
        "return_reason",
        "return_quantity",
        "return_status",
        "condition",
        "return_amount",
        "return_days_after_order"
    )
)

(
    fact_returns.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.fact_returns")
)

print("Created gold.fact_returns")


# ============================================================
# 7. FACT_REFUNDS
# ============================================================

refunds = spark.table(f"{SILVER}.refunds")

fact_refunds = (
    refunds
    .withColumn(
        "refund_processing_hours",
        F.round(
            (
                F.unix_timestamp("refund_processed_at") -
                F.unix_timestamp("refund_requested_at")
            ) / 3600,
            2
        )
    )
    .withColumn(
        "refund_processing_days",
        F.round(
            F.col("refund_processing_hours") / 24,
            2
        )
    )
    .withColumn(
        "sla_breached",
        F.when(
            F.col("refund_processing_days") > 3,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        "refund_id",
        "return_id",
        "order_id",
        "refund_requested_at",
        "refund_processed_at",
        "refund_amount",
        "refund_status",
        "refund_method",
        "refund_processing_hours",
        "refund_processing_days",
        "sla_breached"
    )
)

(
    fact_refunds.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.fact_refunds")
)

print("Created gold.fact_refunds")


# ============================================================
# 8. PRODUCT RETURN METRICS
# ============================================================

product_orders = (
    fact_orders
    .groupBy("product_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_ordered"),
        F.sum("net_order_amount").alias("sales_amount")
    )
)

product_returns = (
    fact_returns
    .groupBy("product_id")
    .agg(
        F.countDistinct("return_id").alias("total_returns"),
        F.sum("return_quantity").alias("units_returned"),
        F.sum("return_amount").alias("returned_amount")
    )
)

product_return_metrics = (
    product_orders
    .join(product_returns, "product_id", "left")
    .fillna(0)
    .join(
        dim_product.select(
            "product_id",
            "product_name",
            "category",
            "subcategory",
            "brand"
        ),
        "product_id",
        "left"
    )
    .withColumn(
        "return_rate",
        F.round(
            F.when(
                F.col("total_orders") > 0,
                F.col("total_returns") /
                F.col("total_orders") * 100
            ).otherwise(0),
            2
        )
    )
    .withColumn(
        "unit_return_rate",
        F.round(
            F.when(
                F.col("units_ordered") > 0,
                F.col("units_returned") /
                F.col("units_ordered") * 100
            ).otherwise(0),
            2
        )
    )
)

(
    product_return_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.product_return_metrics")
)

print("Created gold.product_return_metrics")


# ============================================================
# 9. SELLER RETURN METRICS
# ============================================================

seller_orders = (
    fact_orders
    .groupBy("seller_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_ordered"),
        F.sum("net_order_amount").alias("sales_amount")
    )
)

seller_returns = (
    fact_returns
    .join(
        fact_orders.select(
            "order_id",
            "seller_id"
        ),
        "order_id",
        "left"
    )
    .groupBy("seller_id")
    .agg(
        F.countDistinct("return_id").alias("total_returns"),
        F.sum("return_quantity").alias("units_returned"),
        F.sum("return_amount").alias("returned_amount")
    )
)

seller_return_metrics = (
    seller_orders
    .join(seller_returns, "seller_id", "left")
    .fillna(0)
    .join(
        dim_seller.select(
            "seller_id",
            "seller_name",
            "seller_region",
            "seller_rating"
        ),
        "seller_id",
        "left"
    )
    .withColumn(
        "return_rate",
        F.round(
            F.when(
                F.col("total_orders") > 0,
                F.col("total_returns") /
                F.col("total_orders") * 100
            ).otherwise(0),
            2
        )
    )
)

(
    seller_return_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.seller_return_metrics")
)

print("Created gold.seller_return_metrics")


# ============================================================
# 10. MONTHLY RETURN METRICS
# ============================================================

monthly_orders = (
    fact_orders
    .withColumn(
        "month",
        F.date_trunc("month", "order_date")
    )
    .groupBy("month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_ordered"),
        F.sum("net_order_amount").alias("sales_amount")
    )
)

monthly_returns = (
    fact_returns
    .withColumn(
        "month",
        F.date_trunc("month", "return_date")
    )
    .groupBy("month")
    .agg(
        F.countDistinct("return_id").alias("total_returns"),
        F.sum("return_quantity").alias("units_returned"),
        F.sum("return_amount").alias("returned_amount")
    )
)

monthly_return_metrics = (
    monthly_orders
    .join(monthly_returns, "month", "left")
    .fillna(0)
    .withColumn(
        "return_rate",
        F.round(
            F.when(
                F.col("total_orders") > 0,
                F.col("total_returns") /
                F.col("total_orders") * 100
            ).otherwise(0),
            2
        )
    )
    .orderBy("month")
)

(
    monthly_return_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.monthly_return_metrics")
)

print("Created gold.monthly_return_metrics")


# ============================================================
# 11. REFUND SLA METRICS
# ============================================================

refund_sla_metrics = (
    fact_refunds
    .groupBy(
        F.to_date("refund_requested_at").alias("refund_date")
    )
    .agg(
        F.countDistinct("refund_id").alias("total_refunds"),
        F.sum("refund_amount").alias("total_refund_amount"),
        F.avg("refund_processing_days").alias(
            "avg_processing_days"
        ),
        F.sum(
            F.when(
                F.col("sla_breached") == True,
                1
            ).otherwise(0)
        ).alias("sla_breaches")
    )
    .withColumn(
        "sla_breach_rate",
        F.round(
            F.when(
                F.col("total_refunds") > 0,
                F.col("sla_breaches") /
                F.col("total_refunds") * 100
            ).otherwise(0),
            2
        )
    )
)

(
    refund_sla_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.refund_sla_metrics")
)

print("Created gold.refund_sla_metrics")


# ============================================================
# FINAL GOLD VALIDATION
# ============================================================

print("\n========== GOLD TABLE COUNTS ==========")

gold_tables = [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_date",
    "fact_orders",
    "fact_returns",
    "fact_refunds",
    "product_return_metrics",
    "seller_return_metrics",
    "monthly_return_metrics",
    "refund_sla_metrics"
]

for table_name in gold_tables:

    count = spark.table(
        f"{GOLD}.{table_name}"
    ).count()

    print(f"{table_name}: {count:,} rows")


print("\n========== GOLD LAYER COMPLETE ==========")

In [0]:
from pyspark.sql import functions as F

# Read corrected Silver refunds
refunds = spark.table("ecommerce_returns.silver.refunds")

# Create Gold refund fact table
fact_refunds = (
    refunds
    .withColumn(
        "refund_processing_hours",
        F.round(
            (
                F.unix_timestamp("refund_processed_at") -
                F.unix_timestamp("refund_requested_at")
            ) / 3600,
            2
        )
    )
    .withColumn(
        "refund_processing_days",
        F.round(
            F.col("refund_processing_hours") / 24,
            2
        )
    )
    .withColumn(
        "sla_breached",
        F.when(
            F.col("refund_processing_days") > 3,
            True
        ).otherwise(False)
    )
)

# Replace the old Gold table
(
    fact_refunds.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_returns.gold.fact_refunds")
)

print("Gold fact_refunds rebuilt successfully")
print("Rows:", fact_refunds.count())